In [1]:
import mxnet as mx
import mxnet.ndarray as nd
import numpy as np
import logging
import sys

logging.basicConfig(stream = sys.stdout, level = logging.DEBUG)

np.random.seed(777)

In [2]:
xy = np.loadtxt("data-03-diabetes.csv", delimiter=",", dtype=np.float32)

x_data = xy[:, 0:-1]
y_data = xy[:, [-1]]

train_x_data, test_x_data = x_data[:700], x_data[700:]
train_y_data, test_y_data = y_data[:700], y_data[700:]
print(train_x_data.shape, test_x_data.shape, train_y_data.shape, test_y_data.shape)

# hyper-parameters
sample_num = x_data.shape[0]
dimension = x_data.shape[1]
batch_size = 32

(700, 8) (59, 8) (700, 1) (59, 1)


In [3]:
# Build the symbol
data = mx.sym.Variable("data")
target = mx.sym.Variable("target")
fc = mx.sym.FullyConnected(data=data, num_hidden=1, name='fc')
pred = mx.sym.LogisticRegressionOutput(data=fc, label=target)

In [4]:
# Construct the Module
net = mx.mod.Module(symbol=pred,
                    data_names=['data'],
                    label_names=['target'],
                    context=mx.cpu(0))
net.bind(data_shapes = [mx.io.DataDesc(name='data', shape=(batch_size, dimension), layout='NC')],
         label_shapes= [mx.io.DataDesc(name='target', shape=(batch_size, 1), layout='NC')])
net.init_params(initializer=mx.init.Normal(sigma=0.01))
net.init_optimizer(optimizer='sgd', optimizer_params={'learning_rate': 1E-2, 'momentum': 0.9})

In [5]:
# Training
train_iter = mx.io.NDArrayIter(train_x_data, train_y_data, batch_size, shuffle=True, label_name='target')
metric = mx.metric.CustomMetric(feval=lambda labels, pred: ((pred > 0.5) == labels).mean(),
                                name="acc")
net.fit(train_data=train_iter, eval_metric=metric, num_epoch=15)

INFO:root:Epoch[0] Train-acc=0.600852
INFO:root:Epoch[0] Time cost=0.015
INFO:root:Epoch[1] Train-acc=0.659091
INFO:root:Epoch[1] Time cost=0.027
INFO:root:Epoch[2] Train-acc=0.659091
INFO:root:Epoch[2] Time cost=0.019
INFO:root:Epoch[3] Train-acc=0.659091
INFO:root:Epoch[3] Time cost=0.022
INFO:root:Epoch[4] Train-acc=0.664773
INFO:root:Epoch[4] Time cost=0.018
INFO:root:Epoch[5] Train-acc=0.676136
INFO:root:Epoch[5] Time cost=0.018
INFO:root:Epoch[6] Train-acc=0.697443
INFO:root:Epoch[6] Time cost=0.015
INFO:root:Epoch[7] Train-acc=0.705966
INFO:root:Epoch[7] Time cost=0.013
INFO:root:Epoch[8] Train-acc=0.711648
INFO:root:Epoch[8] Time cost=0.016


C:\app\anaconda3\envs\mxnet\lib\site-packages\mxnet-0.9.5-py3.5.egg\mxnet\module\base_module.py:463: UserWarning: Parameters already initialized and force_init=False. init_params call ignored.
  allow_missing=allow_missing, force_init=force_init)


INFO:root:Epoch[9] Train-acc=0.720170
INFO:root:Epoch[9] Time cost=0.019
INFO:root:Epoch[10] Train-acc=0.737216
INFO:root:Epoch[10] Time cost=0.014
INFO:root:Epoch[11] Train-acc=0.745739
INFO:root:Epoch[11] Time cost=0.019
INFO:root:Epoch[12] Train-acc=0.752841
INFO:root:Epoch[12] Time cost=0.016
INFO:root:Epoch[13] Train-acc=0.759943
INFO:root:Epoch[13] Time cost=0.014
INFO:root:Epoch[14] Train-acc=0.764205
INFO:root:Epoch[14] Time cost=0.019


In [6]:
# Testing
test_iter = mx.io.NDArrayIter(test_x_data, None, batch_size, shuffle=False, label_name=None)

pred_class = (fc > 0)
test_net = mx.mod.Module(symbol=pred_class,
                         data_names=['data'],
                         label_names=None,
                         context=mx.cpu(0))
test_net.bind(data_shapes=[mx.io.DataDesc(name='data', shape=(batch_size, dimension), layout='NC')],
              label_shapes=None,
              for_training=False,
              shared_module=net)
out = test_net.predict(eval_data=test_iter)
print(out.asnumpy())

[[ 1.]
 [ 1.]
 [ 1.]
 [ 0.]
 [ 1.]
 [ 1.]
 [ 0.]
 [ 0.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 0.]
 [ 1.]
 [ 1.]
 [ 0.]
 [ 0.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 0.]
 [ 0.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 0.]
 [ 1.]
 [ 0.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]
 [ 1.]]


In [7]:
# Accuracy
acc = np.equal(test_y_data, out.asnumpy()).mean()
print("Accuracy: %.4g" %acc)

Accuracy: 0.7458
